# 实验二：房源采集与 MySQL 存储

采集 [安居客教学站](http://www.bspider.top/anjuke/) 的房源信息，保存为 CSV，再写入 MySQL。

8 个字段顺序：`title → house → address → struct → area → floor → maketime → price`

> 注意：使用简短 UA（如只写 `Mozilla/5.0`）请求教学站会返回"访问验证"反爬页，
> 因此请求头使用完整的 Chrome 浏览器 User-Agent，并在代码中检查页面标题，一旦命中验证页立即停止采集。

## 第一阶段准备：请求第一页并检查返回内容

In [1]:
import csv
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "http://www.bspider.top/anjuke/"

# 完整浏览器请求头：简短 UA 会被教学站的反爬拦截，返回"访问验证"页
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Referer": "http://www.bspider.top/",
}


def fetch_page(url):
    """请求网页，检查 HTTP 状态与是否为验证页，返回 (状态码, 标题, BeautifulSoup 对象)"""
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")
    page_title = soup.title.get_text(strip=True) if soup.title else ""
    if "访问验证" in page_title:
        raise RuntimeError(f"触发访问验证页，停止采集：{page_title}")
    return r.status_code, page_title, soup


status, page_title, soup = fetch_page(BASE_URL)
print("状态码:", status)
print("网页标题:", page_title)
print("本页房源节点数:", len(soup.select("div.property")))

状态码: 200
网页标题: 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
本页房源节点数: 60


## 第二阶段 1：网页分析与房源解析

Chrome F12 → Elements → Copy selector 复制标题的完整路径后，逐级向上删掉
`#pages-root > div:nth-child(3) > …` 这类按位置选择的长路径，最终找到刚好包住一条房源的
父容器 `div.property`（第二条房源也是相同的 class），在容器内部用相对选择器找字段。

In [2]:
# select() 返回整页所有匹配节点，列表中每一项对应一条房源
rows = soup.select("div.property")
print("div.property 匹配条数:", len(rows))

# select_one() 在一条房源内部返回第一个匹配节点，找不到返回 None
row = rows[0]
node = row.select_one("h3")
print("第一条房源标题:", node.get_text(strip=True))

div.property 匹配条数: 60
第一条房源标题: 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器


In [3]:
# 标题
title = row.select_one("h3").get_text(strip=True)

# 楼盘
house = row.select_one("p.property-content-info-comm-name").get_text(strip=True)

# 地址：文字分散在多个 span 子标签中，get_text() 按原样拼接，不额外插入空格
address = row.select_one("p.property-content-info-comm-address").get_text(strip=True)

# 户型：属性组第一段，同样由多个 span 组成
struct = row.select_one("p.property-content-info-attribute").get_text(strip=True)

# 面积 / 朝向 / 楼层 / 建造年代 混在同一组 p.property-content-info-text 里：
# 先逐项打印确认内容，再按特征取值，避免把朝向当成楼层
items = [p.get_text(strip=True) for p in row.select("p.property-content-info-text")]
items = [t for t in items if t and t != struct]
for i, t in enumerate(items):
    print(f"属性组第{i}项: {t!r}")

area = next((t for t in items if t.endswith("㎡")), "")            # 面积：带 ㎡ 的项（保留单位）
maketime = next((t for t in items if "建造" in t), "")              # 建造年代：含"建造"，缺失填空字符串
floor = next((t for t in items if t not in (area, maketime) and "层" in t), "")  # 楼层：含层号的项
orientation = next((t for t in items if t not in (area, maketime, floor)), "")   # 朝向（不采集）

# 单价：property-price-average 是每平方米单价；总价在 property-price-total（万元），不能取错
price = row.select_one("p.property-price-average").get_text(strip=True)

for name, val in zip(
        ["title", "house", "address", "struct", "area", "floor", "maketime", "price"],
        [title, house, address, struct, area, floor, maketime, price]):
    print(f"{name:9s}= {val}")

属性组第0项: '89.9㎡'
属性组第1项: '南北'
属性组第2项: '低层(共27层)'
属性组第3项: '2019年建造'
title    = 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house    = 万科金域学府翰林菁华里
address  = 南岸学府大道二塘路59号
struct   = 3室2厅1卫
area     = 89.9㎡
floor    = 低层(共27层)
maketime = 2019年建造
price    = 16465元/㎡


In [4]:
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]


def parse_property(row):
    """按固定顺序解析一条房源，返回 8 个字段"""
    h3 = row.select_one("h3")
    title = h3.get_text(strip=True) if h3 else ""

    comm = row.select_one("p.property-content-info-comm-name")
    house = comm.get_text(strip=True) if comm else ""

    addr = row.select_one("p.property-content-info-comm-address")
    address = addr.get_text(strip=True) if addr else ""

    attr = row.select_one("p.property-content-info-attribute")
    struct = attr.get_text(strip=True) if attr else ""

    # 面积/楼层/年代按内容特征取值：节点缺失或顺序改变时不会取错
    items = [p.get_text(strip=True) for p in row.select("p.property-content-info-text")]
    items = [t for t in items if t and t != struct]
    area = next((t for t in items if t.endswith("㎡")), "")
    maketime = next((t for t in items if "建造" in t), "")   # 缺失时填空字符串
    floor = next((t for t in items if t not in (area, maketime) and "层" in t), "")

    avg = row.select_one("p.property-price-average")
    price = avg.get_text(strip=True) if avg else ""

    return [title, house, address, struct, area, floor, maketime, price]


print("第一条  :", parse_property(rows[0]))
print("第二条  :", parse_property(rows[1]))
print("最后一条:", parse_property(rows[-1]))

第一条  : ['万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡']
第二条  : ['黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '渝北渝北黄泥磅紫荆路29号', '3室2厅2卫', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡']
最后一条: ['精装修 电梯房 4室2厅 91平', '华润凯旋天地', '南岸弹子石腾龙大道45号', '4室2厅2卫', '91.3㎡', '低层(共32层)', '2015年建造', '15886元/㎡']


## 第二阶段 2：下一页采集

下一页按钮：`a.next.next-active`，`get("href")` 取地址（只有 `?page=2` 这样的相对值），
用 `urljoin()` 补全；到达末页时按钮被禁用（`next-active` 消失）。

In [5]:
def collect_all(base_url, max_pages=30):
    """翻页采集全部房源。out_list 定义在函数内部，而不是全局。"""
    out_list = []            # 结果列表放在函数内部：每次调用都是新列表，
    visited = {base_url}     # 已访问地址集合，防止重复采集和无限循环
    current_url = base_url
    page_no = 0

    while page_no < max_pages:
        page_no += 1
        _, _, soup_page = fetch_page(current_url)

        page_rows = soup_page.select("div.property")
        if not page_rows:
            raise RuntimeError(f"第{page_no}页未解析到任何房源，结构异常，中止")
        out_list.extend(parse_property(r) for r in page_rows)
        print(f"第{page_no}页 | {current_url} | 本页 {len(page_rows)} 条 | 累计 {len(out_list)} 条")

        if page_no == 2:
            first_titles = {rec[0] for rec in out_list[:60]}
            second_titles = {rec[0] for rec in out_list[60:120]}
            print("    试采检查：第2页与第1页标题完全重复数 =", len(first_titles & second_titles))

        # 停止条件 1：分页区域的“下一页”按钮不存在或已禁用（next-active 消失）
        next_btn = soup_page.select_one("a.next.next-active")
        if next_btn is None:
            print(f"第{page_no}页的“下一页”按钮已禁用，判定到达末页，采集结束")
            break

        # href 原值通常只有 ?page=2 这样的相对值，用 urljoin() 用当前页地址补全
        href = next_btn.get("href")
        next_url = urljoin(current_url, href)
        print(f"    下一页 href 原值: {href} -> urljoin 补全后: {next_url}")

        # 停止条件 2：地址越出教学站，或重复访问（死循环保护）
        if not next_url.startswith(BASE_URL):
            raise RuntimeError(f"下一页地址越出教学站：{next_url}，中止")
        if next_url in visited:
            raise RuntimeError(f"下一页地址重复：{next_url}，可能死循环，中止")
        visited.add(next_url)

        current_url = next_url   # 更新地址，进入下一页（此前漏写导致重复抓第1页）
        time.sleep(1.5)      # 每页之间等待，降低请求频率

    if page_no >= max_pages:
        print(f"达到安全页数上限 {max_pages}，记为中止，不能算采集完成")
    return out_list


all_records = collect_all(BASE_URL)
print("采集总条数（去重前）:", len(all_records))

第1页 | http://www.bspider.top/anjuke/ | 本页 60 条 | 累计 60 条
    下一页 href 原值: ?page=2 -> urljoin 补全后: http://www.bspider.top/anjuke/?page=2


第2页 | http://www.bspider.top/anjuke/?page=2 | 本页 60 条 | 累计 120 条
    试采检查：第2页与第1页标题完全重复数 = 0
    下一页 href 原值: ?page=3 -> urljoin 补全后: http://www.bspider.top/anjuke/?page=3


第3页 | http://www.bspider.top/anjuke/?page=3 | 本页 60 条 | 累计 180 条
    下一页 href 原值: ?page=4 -> urljoin 补全后: http://www.bspider.top/anjuke/?page=4


第4页 | http://www.bspider.top/anjuke/?page=4 | 本页 60 条 | 累计 240 条
    下一页 href 原值: ?page=5 -> urljoin 补全后: http://www.bspider.top/anjuke/?page=5


第5页 | http://www.bspider.top/anjuke/?page=5 | 本页 60 条 | 累计 300 条
    下一页 href 原值: ?page=6 -> urljoin 补全后: http://www.bspider.top/anjuke/?page=6


第6页 | http://www.bspider.top/anjuke/?page=6 | 本页 60 条 | 累计 360 条
    下一页 href 原值: ?page=7 -> urljoin 补全后: http://www.bspider.top/anjuke/?page=7


第7页 | http://www.bspider.top/anjuke/?page=7 | 本页 60 条 | 累计 420 条
    下一页 href 原值: ?page=8 -> urljoin 补全后: http://www.bspider.top/anjuke/?page=8


第8页 | http://www.bspider.top/anjuke/?page=8 | 本页 60 条 | 累计 480 条
    下一页 href 原值: ?page=9 -> urljoin 补全后: http://www.bspider.top/anjuke/?page=9


第9页 | http://www.bspider.top/anjuke/?page=9 | 本页 60 条 | 累计 540 条
    下一页 href 原值: ?page=10 -> urljoin 补全后: http://www.bspider.top/anjuke/?page=10


第10页 | http://www.bspider.top/anjuke/?page=10 | 本页 60 条 | 累计 600 条
第10页的“下一页”按钮已禁用，判定到达末页，采集结束
采集总条数（去重前）: 600


## 第二阶段 3：保存 CSV

In [6]:
# 按 8 个字段完全相同去重（保持原顺序）
seen = set()
unique_records = []
for rec in all_records:
    key = tuple(rec)
    if key not in seen:
        seen.add(key)
        unique_records.append(rec)

dup_count = len(all_records) - len(unique_records)
print(f"去重前 {len(all_records)} 条，完全相同的重复 {dup_count} 条，去重后 {len(unique_records)} 条")

csv_path = Path("data/houseinfo.csv")
csv_path.parent.mkdir(exist_ok=True)
with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:   # utf-8-sig 防 Excel 乱码
    writer = csv.writer(f)
    writer.writerow(FIELDS)              # 表头：与 8 个字段顺序一致
    writer.writerows(unique_records)     # 数据：每条恰好 8 项
print("已保存:", csv_path)

# 用 csv.DictReader 回读检查，表头不计入数据行数
with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == FIELDS, "回读表头顺序不正确"
    back = list(reader)
print("回读数据行数:", len(back))
print("回读首行:", [back[0][k] for k in FIELDS])
print("回读末行:", [back[-1][k] for k in FIELDS])

去重前 600 条，完全相同的重复 102 条，去重后 498 条
已保存: data/houseinfo.csv
回读数据行数: 498
回读首行: ['万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡']
回读末行: ['恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', '恒大照母山山水城', '渝北照母山恒康路9号', '3室2厅1卫', '90.2㎡', '低层(共32层)', '2015年建造', '15530元/㎡']


## 第三阶段：MySQL 数据存储

读取第二阶段的 `data/houseinfo.csv`，写入 MySQL，不重新采集。
连接参数：host=`mysql`（课程数据库服务名），port=3306，database=`smartpark_lab`，
user=`student`，charset=`utf8mb4`，autocommit=False。

In [7]:
# 1. 读取 CSV，检查表头，并按 8 个字段的固定顺序整理入库参数
import pymysql

with open("data/houseinfo.csv", "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == FIELDS, "CSV 表头与预期不一致"
    records = [[row[k] for k in FIELDS] for row in reader]   # 每条记录恰好 8 个值

print("从 CSV 读入记录数:", len(records))
print("首条记录的 8 个参数:")
for name, val in zip(FIELDS, records[0]):
    print(f"  {name:9s}= {val}")

从 CSV 读入记录数: 498
首条记录的 8 个参数:
  title    = 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
  house    = 万科金域学府翰林菁华里
  address  = 南岸学府大道二塘路59号
  struct   = 3室2厅1卫
  area     = 89.9㎡
  floor    = 低层(共27层)
  maketime = 2019年建造
  price    = 16465元/㎡


In [8]:
# 2. 入库程序：executemany() 批量写入，成功 commit()，出错 rollback()，最后关闭
INSERT_SQL = """
INSERT INTO houseinfo
    (title, house, address, struct, area, floor, maketime, price)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""


def import_to_mysql():
    """把 CSV 记录写入 MySQL；按 8 个字段比较库内已有记录，只插入不存在的记录"""
    conn = pymysql.connect(
        host="mysql", port=3306, database="smartpark_lab",
        user="student", password="123456",
        charset="utf8mb4", autocommit=False,   # 由程序显式提交事务
    )
    cursor = conn.cursor()
    try:
        cursor.execute(
            "SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
        existing = {tuple(t) for t in cursor.fetchall()}
        print("入库前条数:", len(existing))

        # 避免重复插入：8 个字段完全相同的记录不再插入（自增长 id 不能防止房源重复）
        new_records = [rec for rec in records if tuple(rec) not in existing]
        print("本次待新增:", len(new_records), "条")

        if new_records:
            affected = cursor.executemany(INSERT_SQL, new_records)
            conn.commit()      # 写入成功后提交
            print(f"executemany() 受影响 {affected} 行，已 commit()")
        else:
            print("没有需要新增的记录，跳过写入")

        cursor.execute("SELECT COUNT(*) FROM houseinfo")
        print("入库后总条数:", cursor.fetchone()[0])
    except Exception as e:
        conn.rollback()        # 出错回滚，撤销本次未提交的修改
        print("写入失败，已 rollback()：", e)
        raise
    finally:
        cursor.close()         # 成功或失败都释放资源：先关游标，再关连接
        conn.close()
        print("游标和连接已关闭")

In [9]:
# 3. 第一次入库（少量检查后写入全部：函数内部已按 8 字段过滤，天然安全）
import_to_mysql()

入库前条数: 0
本次待新增: 498 条


executemany() 受影响 498 行，已 commit()
入库后总条数: 498
游标和连接已关闭


In [10]:
# 4. 关闭上次程序后，同批数据独立重跑：条数不应增加
import_to_mysql()

入库前条数: 498
本次待新增: 0 条
没有需要新增的记录，跳过写入
入库后总条数: 498
游标和连接已关闭


In [11]:
# 5. 核对：总条数、前 5 条样本，并抽 CSV 第一条 / 跨页第 66 条 / 最后一条逐字段核对
conn = pymysql.connect(host="mysql", port=3306, database="smartpark_lab",
                       user="student", password="123456", charset="utf8mb4")
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM houseinfo")
print("数据库总条数:", cursor.fetchone()[0])

cursor.execute("""SELECT title, house, address, struct, area, floor, maketime, price
                  FROM houseinfo ORDER BY id LIMIT 5""")
print("前 5 条样本:")
for t in cursor.fetchall():
    print(" ", t)

print("逐条核对（CSV -> 数据库）:")
for idx in [0, 65, len(records) - 1]:
    page_no = idx // 60 + 1
    rec = records[idx]
    cursor.execute(
        """SELECT title, house, address, struct, area, floor, maketime, price
           FROM houseinfo
           WHERE title=%s AND house=%s AND address=%s AND struct=%s
             AND area=%s AND floor=%s AND maketime=%s AND price=%s""",
        tuple(rec))
    hit = cursor.fetchone()
    verdict = "一致" if hit else "未找到"
    print(f"  样本: 第{page_no}页第{idx % 60 + 1}条 | 匹配: {verdict}")
    print(f"        {rec}")
    if hit:
        for name, a, b in zip(FIELDS, rec, hit):
            if a != b:
                print("        不一致字段:", name, repr(a), repr(b))

cursor.close()
conn.close()
print("核对完成，游标和连接已关闭")

数据库总条数: 498
前 5 条样本:
  ('万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡')
  ('黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '渝北渝北黄泥磅紫荆路29号', '3室2厅2卫', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡')
  ('鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '大雅云居山', '渝北鸳鸯龙展路59号', '3室2厅1卫', '88㎡', '低层(共24层)', '2016年建造', '16487元/㎡')
  ('中交中央公园 横厅精装大两房 出门都是公园', '中交中央公园悦颂', '渝北中央公园翠丹路4号', '2室2厅1卫', '91㎡', '低层(共25层)', '2016年建造', '20660元/㎡')
  ('富力现代广场 豪装大两房 户型方正 看房方便', '富力现代广场', '南岸弹子石弹子石新街59号', '2室2厅1卫', '85.5㎡', '低层(共30层)', '2012年建造', '16969元/㎡')
逐条核对（CSV -> 数据库）:
  样本: 第1页第1条 | 匹配: 一致
        ['万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡']
  样本: 第2页第6条 | 匹配: 一致
        ['照母山金州商圈旁，精装三房！', '恒大世纪城', '渝北照母山湖红路56号', '3室2厅1卫', '94㎡', '低层(共31层)', '2017年建造', '19788元/㎡']
  样本: 第9页第18条 | 匹配: 一致
        ['恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', '恒大照母山山水城', '渝北照母山恒康路9号', '3室2厅1卫', '90.2㎡', '低层(共32层)', 